<a href="https://colab.research.google.com/github/muhammetalicvs-prog/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Signal checks and rule reasoning

1.   Liste öğesi
2.   Liste öğesi



*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline idea

The baseline prioritizes pages that already receive meaningful search visibility and rank within positions where users could reasonably click, but whose observed CTR remains low.

The reasoning is operational rather than causal: a page with sufficient impressions, a useful search position, and weak CTR may be worth reviewing for title, meta description, intent alignment, or search-result presentation. The score does not claim that changing these elements will cause an improvement. It is a decision-support queue for human review.

### Signals to check

I check two signals before encoding the rule:

1. **CTR by position bucket:** This is linked to FlyRank's CTR-fix logic. I expect pages in stronger ranking positions to show higher observed CTR.
2. **Opportunity rate by impression bucket:** This is linked to FlyRank's volume and quick-win logic. I expect higher-volume buckets to contain more measurable review opportunities.

Neither `trend_pct`, `trend_direction`, nor `is_declining_label` is used as a scoring input. These fields are reserved only for retrospective evaluation and leakage checks.


In [18]:
!git clone https://github.com/muhammetalicvs-prog/flyrank-ml.git
%cd flyrank-ml

Cloning into 'flyrank-ml'...
remote: Enumerating objects: 118, done.
remote: Counting objects: 100% (118/118), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 118 (delta 34), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (118/118), 1.86 MiB | 15.91 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/flyrank-ml/flyrank-ml/flyrank-ml


In [19]:
import numpy as np
import pandas as pd
from IPython.display import display

OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print("Available columns:")
print(df.columns.tolist())

display(df.head())

Rows: 30,000
Columns: 44
Available columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [20]:
import pandas as pd

df = pd.read_csv(DATA_PATH)

print("Satır sayısı:", len(df))
print("Sütun sayısı:", df.shape[1])
print(df.columns.tolist())

df.head()


Satır sayısı: 30000
Sütun sayısı: 44
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [21]:
# Signal check 1: CTR by position bucket

signal_1 = df[
    (df["impressions_90d"] > 0)
    & (df["avg_position"] > 0)
    & df["ctr"].notna()
].copy()

position_bins = [0, 3, 10, 20, 50, float("inf")]
position_labels = ["1-3", "4-10", "11-20", "21-50", "51+"]

signal_1["position_bucket"] = pd.cut(
    signal_1["avg_position"],
    bins=position_bins,
    labels=position_labels,
    include_lowest=True
)

position_ctr_table = (
    signal_1
    .groupby("position_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        median_ctr=("ctr", "median"),
        mean_ctr=("ctr", "mean"),
        median_impressions=("impressions_90d", "median"),
    )
    .reset_index()
)

print("Signal 1: CTR by position bucket")
display(position_ctr_table)

Signal 1: CTR by position bucket


,position_bucket,n,median_ctr,mean_ctr,median_impressions
0,1-3,1141,0.00,2.714303,74.0
1,4-10,11842,0.16,0.651045,1184.0
2,11-20,7273,0.10,0.323443,870.0
3,21-50,7225,0.03,0.222345,807.0
4,51+,1314,0.00,0.150784,219.5


### Signal 1 verdict: CONFIRMED

The bucket table generally supports the expected relationship between search position and CTR.

Pages in positions 4–10 have a median CTR of 0.16, compared with 0.10 for positions 11–20, 0.03 for positions 21–50, and 0.00 for positions 51+. Each bucket also contains a visible sample count (`n`), so the result is not based on only a few observations.

The 1–3 bucket does not follow the pattern perfectly because its median CTR is 0.00 despite having 1,141 rows. This suggests that position alone is not enough to define an action. Zero-click pages, query intent, brand effects, and unusual search-result features may influence CTR.

Overall, the result supports checking CTR together with position. The relationship is descriptive rather than causal, so the rule will use both signals only to prioritize human review.

In [22]:
# Signal check 2: Low-CTR opportunity rate by impression bucket

signal_2 = df[
    (df["impressions_90d"] > 0)
    & (df["avg_position"] > 0)
    & df["ctr"].notna()
].copy()

impression_bins = [0, 100, 500, 1000, 5000, float("inf")]
impression_labels = ["1-100", "101-500", "501-1000", "1001-5000", "5001+"]

signal_2["impression_bucket"] = pd.cut(
    signal_2["impressions_90d"],
    bins=impression_bins,
    labels=impression_labels,
    include_lowest=True
)

# Candidate opportunity:
# meaningful visibility + useful position + low CTR
signal_2["is_ctr_opportunity"] = (
    (signal_2["avg_position"] <= 20)
    & (signal_2["ctr"] < 0.20)
)

impression_opportunity_table = (
    signal_2
    .groupby("impression_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        opportunity_count=("is_ctr_opportunity", "sum"),
        opportunity_rate=("is_ctr_opportunity", "mean"),
        median_ctr=("ctr", "median"),
        median_position=("avg_position", "median"),
    )
    .reset_index()
)

impression_opportunity_table["opportunity_rate"] = (
    impression_opportunity_table["opportunity_rate"] * 100
).round(2)

print("Signal 2: CTR opportunity rate by impression bucket")
display(impression_opportunity_table)

Signal 2: CTR opportunity rate by impression bucket


,impression_bucket,n,opportunity_count,opportunity_rate,median_ctr,median_position
0,1-100,6801,4440,65.28,0.00,8.9
1,101-500,5279,1834,34.74,0.00,16.3
2,501-1000,3206,1170,36.49,0.12,15.1
3,1001-5000,7359,2731,37.11,0.16,12.0
4,5001+,6150,1756,28.55,0.22,8.3


### Signal 2 verdict: MIXED

The bucket table does not show a consistent increase in CTR-opportunity rate as impression volume rises.

The 1–100 impression bucket has the highest opportunity rate at 65.28%, while the 5001+ bucket has the lowest rate at 28.55%. The middle-volume buckets are relatively similar, ranging from 34.74% to 37.11%.

This means that higher volume does not automatically imply a higher probability of being a low-CTR opportunity. The unusually high rate in the lowest-volume bucket is likely influenced by many pages with zero observed CTR and limited click evidence.

However, volume still matters operationally. A high-volume page may represent a larger potential impact even when its opportunity rate is lower. Therefore, impression volume will be used as a prioritization weight, not as proof that a page needs action.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 2. Encode one baseline rule and write the ranked queue

The rule prioritizes pages that have meaningful search visibility, rank within positions where clicks are reasonably possible, and still have weak CTR.

The score is intentionally simple and interpretable. It uses only current-window descriptive signals:

- impressions_90d
- ctr
- avg_position

It does not use trend_direction or trend_pct because those fields describe the outcome-like decline behavior and could create leakage.

The rule produces:

- one numeric action score
- one reason code
- one action label
- one ranked review queue

In [23]:
# Baseline rule: high visibility + useful position + weak CTR

queue = df[
    (df["impressions_90d"] > 0)
    & (df["avg_position"] > 0)
    & df["ctr"].notna()
].copy()

# Individual rule components
queue["visibility_score"] = np.clip(
    np.log1p(queue["impressions_90d"]) / np.log1p(10000),
    0,
    1,
)

queue["position_score"] = np.clip(
    (20 - queue["avg_position"]) / 20,
    0,
    1,
)

queue["low_ctr_score"] = np.clip(
    (0.20 - queue["ctr"]) / 0.20,
    0,
    1,
)

# Final interpretable score from 0 to 100
queue["action_score"] = (
    100
    * (
        0.40 * queue["visibility_score"]
        + 0.30 * queue["position_score"]
        + 0.30 * queue["low_ctr_score"]
    )
).round(2)

# One reason code
queue["reason_code"] = "LOW_CTR_WITH_VISIBILITY"

# One action label
queue["action_label"] = "REVIEW_SERP_SNIPPET"

# Keep only genuine rule candidates
queue = queue[
    (queue["impressions_90d"] >= 100)
    & (queue["avg_position"] <= 20)
    & (queue["ctr"] < 0.20)
].copy()

# Rank highest priority first
queue = queue.sort_values(
    by=["action_score", "impressions_90d"],
    ascending=[False, False],
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

queue_columns = [
    "rank",
    "content_id",
    "client_id",
    "action_score",
    "reason_code",
    "action_label",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
]

baseline_queue = queue[queue_columns].copy()

print(f"Queue rows: {len(baseline_queue):,}")
display(baseline_queue.head(10))

Queue rows: 7,497


,rank,content_id,client_id,action_score,reason_code,action_label,impressions_90d,clicks_90d,ctr,avg_position
0,1,content_4a6607efcb46,client_6208ef0f77,95.20,LOW_CTR_WITH_VISIBILITY,REVIEW_SERP_SNIPPET,128068,17,0.01,2.2
1,2,content_134631e65b9e,client_6208ef0f77,94.20,LOW_CTR_WITH_VISIBILITY,REVIEW_SERP_SNIPPET,4417,0,0.00,1.5
2,3,content_c82bc0c24241,client_f369cb89fc,93.55,LOW_CTR_WITH_VISIBILITY,REVIEW_SERP_SNIPPET,13676,0,0.00,4.3
3,4,content_cbdf5a78dcd0,client_19581e27de,93.40,LOW_CTR_WITH_VISIBILITY,REVIEW_SERP_SNIPPET,14830,3,0.02,2.4
4,5,content_c90bfc85694f,client_b4944c6ff0,93.19,LOW_CTR_WITH_VISIBILITY,REVIEW_SERP_SNIPPET,3047,0,0.00,1.1
5,6,content_998f6f88784c,client_f74efabef1,93.10,LOW_CTR_WITH_VISIBILITY,REVIEW_SERP_SNIPPET,12053,3,0.02,2.6
6,7,content_339b357d04c7,client_bbb965ab0c,92.95,LOW_CTR_WITH_VISIBILITY,REVIEW_SERP_SNIPPET,46879,7,0.01,3.7
7,8,content_971e2a5035bc,client_6208ef0f77,92.88,LOW_CTR_WITH_VISIBILITY,REVIEW_SERP_SNIPPET,9179,1,0.01,3.5
8,9,content_50dfd64f9e8e,client_19581e27de,92.88,LOW_CTR_WITH_VISIBILITY,REVIEW_SERP_SNIPPET,4446,0,0.00,2.4
9,10,content_46c51f46d26c,client_7f2253d7e2,92.65,LOW_CTR_WITH_VISIBILITY,REVIEW_SERP_SNIPPET,13341,1,0.01,3.9


In [24]:
# Write the ranked queue

OUTPUT_PATH = OUTPUT_DIR / "baseline_action_score.csv"

baseline_queue.to_csv(OUTPUT_PATH, index=False)

print(f"Queue written to: {OUTPUT_PATH.resolve()}")
print(f"Rows written: {len(baseline_queue):,}")

Queue written to: /content/flyrank-ml/flyrank-ml/flyrank-ml/work/outputs/baseline_action_score.csv
Rows written: 7,497


## 3. Top-10 skeptical review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The top ten rows are reviewed manually below. For each row, I state the recommended action, why the rule ranked it highly, and what evidence could make the recommendation wrong.

This review is important because the score only prioritizes cases. It does not prove that a snippet change will improve performance.

### Rank 1 — content_4a6607efcb46

**Action:** Review the SERP snippet.

**Why it is here:** The page has 128,068 impressions, an average position of 2.2, and a CTR of only 0.01. It combines exceptionally high visibility with a very strong ranking position and almost no clicks.

**What would make it wrong:** The impressions may come from navigational or zero-click queries, SERP features may answer the query directly, or the reported position and CTR may aggregate unrelated queries with very different intent.

---

### Rank 2 — content_134631e65b9e

**Action:** Review the SERP snippet.

**Why it is here:** The page ranks at an average position of 1.5 and has 4,417 impressions, but records no clicks and a CTR of 0.00.

**What would make it wrong:** The page may appear for queries where users do not need to click, the page could be an image or rich-result source, or the click tracking may be incomplete.

---

### Rank 3 — content_c82bc0c24241

**Action:** Review the SERP snippet.

**Why it is here:** The page has 13,676 impressions, an average position of 4.3, and no recorded clicks. This is a strong visibility signal paired with zero CTR.

**What would make it wrong:** The page may rank for irrelevant broad queries, impressions may be concentrated in low-intent searches, or the title and description may not be the true cause of low CTR.

---

### Rank 4 — content_cbdf5a78dcd0

**Action:** Review the SERP snippet.

**Why it is here:** The page has 14,830 impressions, ranks at 2.4 on average, and has only 3 clicks with a CTR of 0.02.

**What would make it wrong:** The page may be shown in a SERP layout dominated by ads, maps, or featured snippets, or the average position may hide weaker positions across most queries.

---

### Rank 5 — content_c90bfc85694f

**Action:** Review the SERP snippet.

**Why it is here:** The page ranks at an average position of 1.1 and has 3,047 impressions, but no recorded clicks.

**What would make it wrong:** The result may satisfy users without a click, the query may be highly navigational toward another brand, or the data may contain reporting or attribution problems.

---

### Rank 6 — content_998f6f88784c

**Action:** Review the SERP snippet.

**Why it is here:** The page has 12,053 impressions, an average position of 2.6, and only 3 clicks, resulting in a CTR of 0.02.

**What would make it wrong:** The page may rank for many loosely related queries, search-result features may reduce organic clicks, or the low CTR may reflect intent mismatch rather than a weak snippet.

---

### Rank 7 — content_339b357d04c7

**Action:** Review the SERP snippet.

**Why it is here:** The page has 46,879 impressions, ranks at 3.7 on average, and has only 7 clicks with a CTR of 0.01.

**What would make it wrong:** A small number of very high-impression queries may dominate the aggregate, users may prefer another result type, or the page may not be eligible for a meaningful snippet improvement.

---

### Rank 8 — content_971e2a5035bc

**Action:** Review the SERP snippet.

**Why it is here:** The page has 9,179 impressions, an average position of 3.5, and only 1 click, producing a CTR of 0.01.

**What would make it wrong:** The page may appear for informational searches resolved directly on the SERP, the query mix may be irrelevant, or the position metric may not reflect the most important queries.

---

### Rank 9 — content_50dfd64f9e8e

**Action:** Review the SERP snippet.

**Why it is here:** The page has 4,446 impressions, ranks at 2.4 on average, and has zero clicks.

**What would make it wrong:** The result may be affected by tracking gaps, rich-result competition, branded-query behavior, or a mismatch between the page and the queries generating impressions.

---

### Rank 10 — content_46c51f46d26c

**Action:** Review the SERP snippet.

**Why it is here:** The page has 13,341 impressions, an average position of 3.9, and only 1 click with a CTR of 0.01.

**What would make it wrong:** The page may rank for irrelevant or low-click-intent queries, the SERP may contain strong competing features, or the snippet may already be appropriate and the real issue may be search intent.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
## 4. Weak picks and rule limitations

In [25]:
# Review weaker candidates near the bottom of the action queue

weak_picks = queue.tail(5)

display(weak_picks)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,position_tier,trend_direction,trend_pct,visibility_score,position_score,low_ctr_score,action_score,reason_code,action_label,rank
7492,content_530aaef6f9a7,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,3469.0,21032.0,...,striking,down,-34.7,0.685083,0.005,0.10,30.55,LOW_CTR_WITH_VISIBILITY,REVIEW_SERP_SNIPPET,7493
7493,content_c6bf8be3e38a,client_4e07408562,30.0,0.46,MEDIUM,0.84,keyword article,commercial,3235.0,19769.0,...,striking,down,-28.4,0.678993,0.060,0.05,30.46,LOW_CTR_WITH_VISIBILITY,REVIEW_SERP_SNIPPET,7494
7494,content_711fce0165b4,client_a88a7902cb,320.0,1.00,HIGH,0.55,keyword article,transactional,3058.0,20155.0,...,striking,up,184.8,0.677733,0.045,0.05,29.96,LOW_CTR_WITH_VISIBILITY,REVIEW_SERP_SNIPPET,7495
7495,content_fce268398d80,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,5113.0,33240.0,...,striking,stable,17.2,0.682890,0.025,0.05,29.57,LOW_CTR_WITH_VISIBILITY,REVIEW_SERP_SNIPPET,7496
7496,content_cacad3ba234c,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,3505.0,21688.0,...,striking,down,-49.2,0.677733,0.020,0.05,29.21,LOW_CTR_WITH_VISIBILITY,REVIEW_SERP_SNIPPET,7497


### Weak-pick review

The bottom five rows still satisfy the baseline rule, but their action scores are much lower than the top-ranked cases.

These rows appear weaker because their visibility, position, and low-CTR components produce smaller combined scores. They may still deserve review, but they should not be treated with the same urgency as the top ten.

This comparison shows that the baseline is useful as a ranking tool rather than as a simple yes-or-no decision rule.

The bottom rows also reveal an important limitation: the exported queue still contains fields such as `trend_direction` and `trend_pct`. These fields were not used to calculate the score, but they should be removed from the final exported queue to avoid confusion about leakage.

In [26]:
# Keep only safe and useful columns in the final ranked queue

safe_queue_columns = [
    "rank",
    "content_id",
    "client_id",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "visibility_score",
    "position_score",
    "low_ctr_score",
    "action_score",
    "reason_code",
    "action_label",
]

final_queue = queue[safe_queue_columns].copy()

display(final_queue.head(10))
print("Final queue rows:", len(final_queue))

,rank,content_id,client_id,impressions_90d,clicks_90d,ctr,avg_position,visibility_score,position_score,low_ctr_score,action_score,reason_code,action_label
0,1,content_4a6607efcb46,client_6208ef0f77,128068,17,0.01,2.2,1.000000,0.890,0.95,95.20,LOW_CTR_WITH_VISIBILITY,REVIEW_SERP_SNIPPET
1,2,content_134631e65b9e,client_6208ef0f77,4417,0,0.00,1.5,0.911297,0.925,1.00,94.20,LOW_CTR_WITH_VISIBILITY,REVIEW_SERP_SNIPPET
2,3,content_c82bc0c24241,client_f369cb89fc,13676,0,0.00,4.3,1.000000,0.785,1.00,93.55,LOW_CTR_WITH_VISIBILITY,REVIEW_SERP_SNIPPET
3,4,content_cbdf5a78dcd0,client_19581e27de,14830,3,0.02,2.4,1.000000,0.880,0.90,93.40,LOW_CTR_WITH_VISIBILITY,REVIEW_SERP_SNIPPET
4,5,content_c90bfc85694f,client_b4944c6ff0,3047,0,0.00,1.1,0.870994,0.945,1.00,93.19,LOW_CTR_WITH_VISIBILITY,REVIEW_SERP_SNIPPET
5,6,content_998f6f88784c,client_f74efabef1,12053,3,0.02,2.6,1.000000,0.870,0.90,93.10,LOW_CTR_WITH_VISIBILITY,REVIEW_SERP_SNIPPET
6,7,content_339b357d04c7,client_bbb965ab0c,46879,7,0.01,3.7,1.000000,0.815,0.95,92.95,LOW_CTR_WITH_VISIBILITY,REVIEW_SERP_SNIPPET
7,8,content_971e2a5035bc,client_6208ef0f77,9179,1,0.01,3.5,0.990700,0.825,0.95,92.88,LOW_CTR_WITH_VISIBILITY,REVIEW_SERP_SNIPPET
8,9,content_50dfd64f9e8e,client_19581e27de,4446,0,0.00,2.4,0.912007,0.880,1.00,92.88,LOW_CTR_WITH_VISIBILITY,REVIEW_SERP_SNIPPET
9,10,content_46c51f46d26c,client_7f2253d7e2,13341,1,0.01,3.9,1.000000,0.805,0.95,92.65,LOW_CTR_WITH_VISIBILITY,REVIEW_SERP_SNIPPET


Final queue rows: 7497


In [27]:
OUTPUT_PATH = OUTPUT_DIR / "baseline_action_score.csv"

final_queue.to_csv(OUTPUT_PATH, index=False)

print("Queue written to:", OUTPUT_PATH)
print("File exists:", OUTPUT_PATH.exists())

Queue written to: work/outputs/baseline_action_score.csv
File exists: True


The rows near the bottom of the action queue still satisfy the rule, but they are weaker priorities than the top-ranked pages.

They generally have lower impression volume, weaker ranking positions, smaller CTR gaps, or a combination of these factors. This shows that the score is useful for ordering candidates rather than making a binary claim that every flagged page requires immediate action.

Potential weaknesses of the baseline include:

- Average position and CTR are aggregated across multiple queries.
- Low CTR does not prove that the SERP snippet is the cause.
- Search intent, branded demand, ads, featured snippets, and other SERP features are not represented.
- The thresholds are hand-written and may not transfer equally across clients.
- The score does not estimate causal impact or expected uplift.

The queue should therefore be treated as decision support for human review, not as an automatic recommendation system.

## 5. Self-check

The checks below confirm that the final queue is ranked correctly, contains the required baseline outputs, excludes leakage-related fields, and can regenerate the required CSV.

In [28]:
# Final self-check

required_output_columns = {
    "rank",
    "content_id",
    "client_id",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "action_score",
    "reason_code",
    "action_label",
}

missing_output_columns = required_output_columns.difference(
    final_queue.columns
)

assert not missing_output_columns, (
    f"Missing output columns: {sorted(missing_output_columns)}"
)

assert final_queue["action_score"].is_monotonic_decreasing
assert final_queue["rank"].tolist() == list(
    range(1, len(final_queue) + 1)
)

assert final_queue["reason_code"].nunique() == 1
assert final_queue["action_label"].nunique() == 1

for forbidden_column in [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
]:
    assert forbidden_column not in final_queue.columns

assert OUTPUT_PATH.exists()
assert len(final_queue) > 0

print("Self-check passed.")
print("Final queue rows:", len(final_queue))
print("Reason code:", final_queue["reason_code"].unique().tolist())
print("Action label:", final_queue["action_label"].unique().tolist())
print("Output file:", OUTPUT_PATH)

Self-check passed.
Final queue rows: 7497
Reason code: ['LOW_CTR_WITH_VISIBILITY']
Action label: ['REVIEW_SERP_SNIPPET']
Output file: work/outputs/baseline_action_score.csv


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.